In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

In [ ]:
df=pd.read_csv("data-2.csv")
df.drop(["Unnamed: 32"],axis=1,inplace=True)

In [ ]:
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 


X=df.iloc[:,2:32]
y=df.iloc[:,1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,shuffle=True)


scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [ ]:
display(X.head())

In [ ]:
display(y.head())

In [ ]:
df.diagnosis.replace({"M":1,"B":0},inplace=True)
df.diagnosis.unique()

In [ ]:
import seaborn as sns
sns.countplot('diagnosis', data=df)

In [ ]:
! pip install np
! pip install sklearn
! pip install sklearn_nature_inspired_algorithms==0.4.3

In [ ]:
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn_nature_inspired_algorithms.model_selection.nature_inspired_search_cv import NatureInspiredSearchCV
from sklearn_nature_inspired_algorithms.helpers import score_by_generation_lineplot

In [ ]:
param_grid = { 
    'n_estimators': range(20, 400, 20), 
    'max_depth': range(5, 300, 20),
    'min_samples_split': range(2, 50, 5),
    'max_features': ["auto", "sqrt", "log2"],
}

In [ ]:
import numpy as np

print(f'This is {np.multiply.reduce([len(param) for param in param_grid.values()])} combinations in total.')

In [ ]:
%%time

clf = RandomForestClassifier()

nia_search = NatureInspiredSearchCV(
    clf,
    param_grid,
    cv=3,
    verbose=0,
    algorithm='fa',
    population_size=50,
    max_n_gen=100,
    max_stagnating_gen=8,
    runs=6,
    n_jobs=-1,
    scoring='f1_macro')

nia_search.fit(X_train, y_train)

In [ ]:
score_by_generation_lineplot(nia_search, metric='mean')

In [ ]:
from sklearn_nature_inspired_algorithms.helpers import score_by_generation_violinplot
score_by_generation_violinplot(nia_search, run=9)

In [ ]:
nia_search.best_params_

In [ ]:
clf = RandomForestClassifier(**nia_search.best_params_)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))

In [ ]:
from sklearn.metrics import accuracy_score as acs
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_recall_fscore_support as score
import matplotlib.pyplot as plt

cm=confusion_matrix(y_pred,y_test)
class_label = ["malignant", "benign"]
df_cm = pd.DataFrame(cm, index=class_label,columns=class_label)

precision, recall, fscore, train_support = score(y_test, y_pred, pos_label="M", average='binary')
print('Precision: {} \nRecall: {} \nF1-Score: {} \nAccuracy: {}'.format(
    round(precision, 3), round(recall, 3), round(fscore,3), round(acs(y_test,y_pred), 4)*100) +"% \n")

sns.heatmap(df_cm,annot=True,cmap='jet',linewidths=2,fmt='d')
plt.title("Confusion Matrix",fontsize=15)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()